# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 14 · Passing-line feature research

Round 5 failed temporal replication. Preserve its control and stop the full direct-state addition. This notebook adds **21 pass-axis state fields and 27 exact-offset history fields**, not another direct-state bundle. It does not train models.

The first three figures use the uploaded Round 5 aggregate report. New diagnostics require your own run. No synthetic accuracy results are stored here.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round6')
OUT = Path('/home/sagemaker-user/nfl-feature-round6-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract Round 6 first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=2):
    cmd = [str(PY), str(KIT/'run_round.py'), stage, '--fold', str(fold)]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        process.wait(timeout=10)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve checkpoints; export the report. Do not retry an unchanged failure.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Diagnose the prior result
Both later folds worsened. The role and horizon slices are descriptive; they must not be used to switch models by role or drop long forecasts.

In [ ]:
show(visuals.previous_folds(KIT), 'round5_folds')
show(visuals.previous_horizon(KIT), 'round5_horizons')
show(visuals.previous_roles(KIT), 'round5_roles')

## Verify sources and replay the existing models
Read-only replay of eight Round 5 coordinate models. No old models are refitted, no packages are downloaded, and no previous receipts are rewritten. Expected: `pass_axis_preflight_passed`.

In [ ]:
run('preflight')
show(visuals.populations(OUT), 'fold_populations')

## Training-only smoke: 32 plays
Features use the passer position at the same observation time as the player. Missing passer/receiver observations are masked, not carried forward. Earlier endpoints use exact 5-, 10-, and 15-frame offsets. This first stage also independently verifies the historical float32/float64 storage parity. Expected: `pass_axis_smoke_passed`.

In [ ]:
run('smoke')
show(visuals.support(OUT), 'measurement_support')

## Prepare the frozen 1,024 plays
Run only after the smoke passes. Existing 32-play checkpoints are reused. No new play selection or raw output CSV read is performed. Expected: `pass_axis_features_ready`. No model fit occurs in notebook 14.

In [ ]:
run('prepare')
result = json.loads((OUT/'preparation.json').read_text())
print(json.dumps(result, indent=2))

## Save and proceed
Save with Ctrl+S. Open `15_pass_axis_ablation.ipynb` only after preparation passes. Keep all historical artifacts. Do not run another round or a terminal stage concurrently. Research rationale, limits, and the exact comparisons are in `EXPERIMENT_PROTOCOL.md`.